## Reusable Claims transformation

I keep the transformation itself separate from the Lakeflow dataset
definition so the transformation logic remains easy to read and maintain.

The transformation performs:

1. column-name standardization
2. explicit data typing
3. categorical cleanup
4. duplicate Claim reconciliation
5. fraud-label conflict handling
6. analytical enrichment
7. Silver lineage enrichment

Source profiling identified duplicate `Claim_ID` records.

The duplicated Claims contain the same business attributes. Some are exact
duplicates, while others disagree only on the historical `Is_Fraudulent`
label.

I preserve all raw records in Bronze and resolve this source-quality issue in
Silver.

When duplicate Claim records contain conflicting fraud labels, I retain one
Claim, set `is_fraudulent` to null, and set `fraud_label_conflict = true`
rather than arbitrarily selecting one label.


In [0]:
#  importing the Lakeflow API and Spark functions used by this transformation.

from pyspark import pipelines as dp
from pyspark.sql import functions as F

CATALOG = "health_insurance"

SOURCE_TABLE = f"{CATALOG}.bronze.claims_raw"
QUALITY_RULES_TABLE = f"{CATALOG}.governance.quality_rules"


In [0]:
# loading active quality rules from the governed Unity Catalog repository.

def get_quality_rules(dataset, severity):
    rows = (
        spark.read
        .table(QUALITY_RULES_TABLE)
        .filter(
            (F.col("dataset") == dataset)
            & (F.col("severity") == severity)
            & F.col("is_active")
        )
        .select(
            "rule_name",
            "constraint"
        )
        .collect()
    )

    return {
        row["rule_name"]: row["constraint"]
        for row in rows
    }


CLAIMS_WARN_RULES = get_quality_rules("claims", "WARN")
CLAIMS_DROP_RULES = get_quality_rules("claims", "DROP")
CLAIMS_FAIL_RULES = get_quality_rules("claims", "FAIL")


## Reusable Claims transformation

I keep the transformation itself separate from the Lakeflow dataset
definition so the transformation logic remains easy to read and test.

The function performs the same Bronze-to-Silver work I validated earlier:
column standardization, explicit typing, categorical cleanup, derived
attributes, and lineage enrichment.


In [0]:
# applying the validated Bronze-to-Silver Claims transformation.

def transform_claims(claims_bronze_df):

    claims_standardized_df = (
        claims_bronze_df

        .withColumnRenamed("Patient_ID", "patient_id")
        .withColumnRenamed("Policy_Number", "policy_number")
        .withColumnRenamed("Claim_ID", "claim_id")
        .withColumnRenamed("Claim_Date", "claim_date")
        .withColumnRenamed("Service_Date", "service_date")
        .withColumnRenamed(
            "Policy_Expiration_Date",
            "policy_expiration_date"
        )
        .withColumnRenamed("Claim_Amount", "claim_amount")
        .withColumnRenamed("Patient_Age", "patient_age")
        .withColumnRenamed("Patient_Gender", "patient_gender")
        .withColumnRenamed("Patient_City", "patient_city")
        .withColumnRenamed("Patient_State", "patient_state")
        .withColumnRenamed("Hospital_ID", "hospital_id")
        .withColumnRenamed("Provider_Type", "provider_type")
        .withColumnRenamed(
            "Provider_Specialty",
            "provider_specialty"
        )
        .withColumnRenamed("Provider_City", "provider_city")
        .withColumnRenamed("Provider_State", "provider_state")
        .withColumnRenamed("Diagnosis_Code", "diagnosis_code")
        .withColumnRenamed("Procedure_Code", "procedure_code")
        .withColumnRenamed(
            "Number_of_Procedures",
            "number_of_procedures"
        )
        .withColumnRenamed("Admission_Type", "admission_type")
        .withColumnRenamed("Discharge_Type", "discharge_type")
        .withColumnRenamed(
            "Length_of_Stay_Days",
            "length_of_stay_days"
        )
        .withColumnRenamed("Service_Type", "service_type")
        .withColumnRenamed(
            "Deductible_Amount",
            "deductible_amount"
        )
        .withColumnRenamed("CoPay_Amount", "copay_amount")
        .withColumnRenamed(
            "Number_of_Previous_Claims_Patient",
            "previous_claims_patient"
        )
        .withColumnRenamed(
            "Number_of_Previous_Claims_Provider",
            "previous_claims_provider"
        )
        .withColumnRenamed(
            "Provider_Patient_Distance_Miles",
            "provider_patient_distance_miles"
        )
        .withColumnRenamed(
            "Claim_Submitted_Late",
            "claim_submitted_late"
        )
        .withColumnRenamed(
            "Is_Fraudulent",
            "is_fraudulent"
        )
    )

    claims_typed_df = (
        claims_standardized_df

        .withColumn(
            "patient_id",
            F.col("patient_id").cast("long")
        )
        .withColumn(
            "claim_id",
            F.col("claim_id").cast("long")
        )
        .withColumn(
            "hospital_id",
            F.col("hospital_id").cast("long")
        )
        .withColumn(
            "claim_date",
            F.to_date("claim_date")
        )
        .withColumn(
            "service_date",
            F.to_date("service_date")
        )
        .withColumn(
            "policy_expiration_date",
            F.to_date("policy_expiration_date")
        )
        .withColumn(
            "claim_amount",
            F.col("claim_amount").cast("decimal(18,2)")
        )
        .withColumn(
            "deductible_amount",
            F.col("deductible_amount").cast("decimal(18,2)")
        )
        .withColumn(
            "copay_amount",
            F.col("copay_amount").cast("decimal(18,2)")
        )
        .withColumn(
            "provider_patient_distance_miles",
            F.col("provider_patient_distance_miles").cast("double")
        )
        .withColumn(
            "patient_age",
            F.col("patient_age").cast("int")
        )
        .withColumn(
            "number_of_procedures",
            F.col("number_of_procedures").cast("int")
        )
        .withColumn(
            "length_of_stay_days",
            F.col("length_of_stay_days").cast("int")
        )
        .withColumn(
            "previous_claims_patient",
            F.col("previous_claims_patient").cast("int")
        )
        .withColumn(
            "previous_claims_provider",
            F.col("previous_claims_provider").cast("int")
        )
        .withColumn(
            "claim_submitted_late",
            F.col("claim_submitted_late").cast("boolean")
        )
        .withColumn(
            "is_fraudulent",
            F.col("is_fraudulent").cast("boolean")
        )
    )

    claims_clean_df = (
        claims_typed_df

        .withColumn(
            "patient_gender",
            F.upper(F.trim("patient_gender"))
        )
        .withColumn(
            "patient_city",
            F.initcap(F.trim("patient_city"))
        )
        .withColumn(
            "patient_state",
            F.upper(F.trim("patient_state"))
        )
        .withColumn(
            "provider_type",
            F.upper(F.trim("provider_type"))
        )
        .withColumn(
            "provider_specialty",
            F.initcap(F.trim("provider_specialty"))
        )
        .withColumn(
            "provider_city",
            F.initcap(F.trim("provider_city"))
        )
        .withColumn(
            "provider_state",
            F.upper(F.trim("provider_state"))
        )
        .withColumn(
            "admission_type",
            F.upper(F.trim("admission_type"))
        )
        .withColumn(
            "discharge_type",
            F.upper(F.trim("discharge_type"))
        )
        .withColumn(
            "service_type",
            F.upper(F.trim("service_type"))
        )
    )
    # reconciling duplicate source Claims before analytical enrichment.

    claim_business_columns = [
        column_name
        for column_name in claims_clean_df.columns
        if not column_name.startswith("_")
        and column_name != "is_fraudulent"
    ]

    claims_reconciled_df = (
        claims_clean_df

        .groupBy(*claim_business_columns)

        .agg(
            # I am recording how many Bronze rows represented this Claim.
            F.count(F.lit(1))
            .cast("int")
            .alias("_source_record_count"),

            # I am measuring whether multiple fraud labels were supplied.
            F.countDistinct("is_fraudulent")
            .cast("int")
            .alias("_fraud_label_distinct_count"),

            # I am retaining the label when there is only one known value.
            F.first(
                "is_fraudulent",
                ignorenulls=True
            ).alias("_fraud_label_value"),

            # I am retaining technical source lineage.
            F.max("_ingested_at")
            .alias("_ingested_at"),

            F.first(
                "_source_system",
                ignorenulls=True
            ).alias("_source_system"),

            F.first(
                "_source_file",
                ignorenulls=True
            ).alias("_source_file")
        )

        .withColumn(
            "fraud_label_conflict",
            F.col("_fraud_label_distinct_count") > 1
        )

        .withColumn(
            "is_fraudulent",
            F.when(
                F.col("fraud_label_conflict"),
                F.lit(None).cast("boolean")
            )
            .otherwise(
                F.col("_fraud_label_value")
            )
        )

        .drop(
            "_fraud_label_distinct_count",
            "_fraud_label_value"
        )
    )
    claims_enriched_df = (
        claims_reconciled_df

        .withColumn(
            "claim_submission_delay_days",
            F.datediff(
                F.col("claim_date"),
                F.col("service_date")
            )
        )

        .withColumn(
            "claim_amount_band",
            F.when(
                F.col("claim_amount") < 1000,
                "LOW"
            )
            .when(
                F.col("claim_amount") < 5000,
                "MEDIUM"
            )
            .when(
                F.col("claim_amount") < 10000,
                "HIGH"
            )
            .otherwise("VERY_HIGH")
        )

        .withColumn(
            "patient_age_group",
            F.when(
                F.col("patient_age") < 18,
                "UNDER_18"
            )
            .when(
                F.col("patient_age") < 35,
                "18_34"
            )
            .when(
                F.col("patient_age") < 50,
                "35_49"
            )
            .when(
                F.col("patient_age") < 65,
                "50_64"
            )
            .otherwise("65_PLUS")
        )
    )

    claims_silver_df = (
        claims_enriched_df
        .withColumn(
            "_silver_transformed_at",
            F.current_timestamp()
        )
    )

    return claims_silver_df


## Lakeflow-managed Claims Silver dataset

I apply the centrally governed quality rules to the transformed Claims
dataset.

- WARN rules keep the row and record quality metrics.
- DROP rules prevent unusable records from reaching Silver.
- FAIL rules stop this Claims flow when a critical contract is violated.

Because the current Kaggle Bronze source is a batch snapshot, I use a
materialized view with a batch read.


In [0]:
# I am defining the pipeline-managed Claims Silver materialized view.

@dp.materialized_view(
    name="claims",
    comment="Validated and standardized Kaggle health insurance claims."
)
@dp.expect_all(CLAIMS_WARN_RULES)
@dp.expect_all_or_drop(CLAIMS_DROP_RULES)
@dp.expect_all_or_fail(CLAIMS_FAIL_RULES)
def claims():
    claims_bronze_df = spark.read.table(SOURCE_TABLE)

    return transform_claims(
        claims_bronze_df
    )


## Pipeline result

This notebook no longer performs manual Silver persistence or notebook-level
row reconciliation.

When it is added as a source to the Claims Silver Lakeflow pipeline:

1. Lakeflow reads the Bronze Claims snapshot.
2. I apply the previously validated Claims transformation.
3. Lakeflow evaluates the active governed quality rules.
4. Lakeflow manages the `claims` Silver materialized view.
5. Expectation results are available through pipeline monitoring and the
   event log.

The development-only `display()`, `count()`, schema inspection, direct
`saveAsTable()`, and post-write verification cells have been removed from
the pipeline execution path.
